# Thesis final model comparison

This notebook produces only the final thesis artifacts requested for the model comparison section:

1. 1000-epoch model comparison: MSE table, L2 table, training loss history, and corrected field triptychs.
2. 2000-epoch baseline vs dynamic PINN comparison: MSE table, L2 table, training loss history, corrected field triptychs, amplitude comparison, and spatial solution slices.
3. Dynamic PINN vs uploaded checkpoint: L2 table only.

All figures are saved as `.pgf`; tables are saved as `.tex`.

In [ ]:
from __future__ import annotations

import json
import math
import os
import shutil
import sys
from collections import OrderedDict
from pathlib import Path
from typing import Any

# Keep matplotlib cache out of the home directory on locked-down systems.
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-pinn-swe')

import numpy as np
import pandas as pd
import torch
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print
from matplotlib import colors
import matplotlib as mpl
import matplotlib.pyplot as plt
from torch import nn


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training_results').is_dir():
            return candidate
    raise RuntimeError('Cannot locate project root. Launch from the repo root or a notebook subdirectory.')


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

TRAINING_RESULTS_DIR = PROJECT_ROOT / 'training_results'
OUTPUT_DIR = PROJECT_ROOT / 'analysis_outputs' / 'thesis_final_model_comparison'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for directory in [TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

LATEX_ENGINE = os.environ.get('PINN_SWE_LATEX_ENGINE', 'xelatex')
mpl.rcParams.update({
    'pgf.texsystem': LATEX_ENGINE,
    'font.family': 'serif',
    'text.usetex': False,
    'pgf.rcfonts': False,
    'pgf.preamble': '\n'.join([
        r'\usepackage{fontspec}',
        r'\usepackage{polyglossia}',
        r'\setmainfont{Nimbus Roman}',
        r'\newfontfamily\cyrillicfont{Nimbus Roman}',
        r'\setmainlanguage{russian}',
    ]),
    'axes.unicode_minus': False,
})

RUNS_1K = OrderedDict([
    ('baseline', 'Исходная PINN'),
    ('rad_k1_c1', 'PINN с RAD'),
    ('rar_d_k1_c1', 'PINN с RAR-D'),
    ('euler_transition_w3e6', 'PINN с эйлеровой компонентой'),
    ('integral_conservation_anchor_w1', 'PINN с интегральными ограничениями'),
    ('swe_dynamics_modal_w1_dt05d', 'PINN с модальной динамической компонентой'),
])

RUNS_2K = OrderedDict([
    ('baseline_2k_epoch', 'Исходная PINN'),
    ('swe_dynamics_modal_w1_dt05d_2k_epoch', 'PINN с модальной динамической компонентой'),
])

DYNAMIC_2K_RUN = 'swe_dynamics_modal_w1_dt05d_2k_epoch'
ARTICLE_RUN_DIR = TRAINING_RESULTS_DIR / 'artical_pinn'
ARTICLE_EVAL_BATCH_SIZE = 131072
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Project root: {PROJECT_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'LaTeX engine for PGF export: {LATEX_ENGINE}')
if shutil.which(LATEX_ENGINE) is None:
    print(f'WARNING: {LATEX_ENGINE} was not found. PGF figure export will fail until a TeX engine is installed.')
print(f'Checkpoint directory exists: {ARTICLE_RUN_DIR.exists()}')
print(f'Evaluation device: {DEVICE}')

## Helpers

In [ ]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def load_npy(run_dir: Path, name: str, required: bool = True) -> np.ndarray | None:
    path = run_dir / name
    if path.exists():
        return np.load(path, allow_pickle=False)
    if required:
        raise FileNotFoundError(path)
    return None


def safe_torch_load(path: Path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception:
        return torch.load(path, map_location=map_location, weights_only=False)


def infer_epoch_axis(history: np.ndarray, meta: dict[str, Any]) -> np.ndarray:
    output_period = int(meta.get('console_output_period') or meta.get('output_period') or 1)
    epochs = np.arange(len(history), dtype=float) * output_period
    if len(epochs) > 1 and meta.get('epochs') is not None:
        epochs[-1] = min(float(meta['epochs']), epochs[-1])
    return epochs


def relative_l2(pred: np.ndarray, exact: np.ndarray) -> float:
    denom = float(np.linalg.norm(exact.ravel()))
    err = float(np.linalg.norm((pred - exact).ravel()))
    return err / denom if denom else err


def relative_l2_after_time(
    pred: np.ndarray,
    exact: np.ndarray,
    time_mesh: np.ndarray,
    minimum_time_seconds: float,
) -> float:
    mask = time_mesh[0, :] >= minimum_time_seconds
    return relative_l2(pred[:, mask], exact[:, mask])


def combined_relative_l2(pred_eta: np.ndarray, exact_eta: np.ndarray, pred_u: np.ndarray, exact_u: np.ndarray) -> float:
    err_norm_sq = np.linalg.norm((pred_eta - exact_eta).ravel()) ** 2 + np.linalg.norm((pred_u - exact_u).ravel()) ** 2
    ref_norm_sq = np.linalg.norm(exact_eta.ravel()) ** 2 + np.linalg.norm(exact_u.ravel()) ** 2
    return float(math.sqrt(err_norm_sq) / math.sqrt(ref_norm_sq))


def load_run_fields(run_name: str) -> dict[str, np.ndarray]:
    run_dir = TRAINING_RESULTS_DIR / run_name
    return {
        'exact_eta': load_npy(run_dir, 'exact_solution_h_values.npy'),
        'exact_u': load_npy(run_dir, 'exact_solution_u_values.npy'),
        'pred_eta': load_npy(run_dir, 'dimensional_network_output_h_values.npy'),
        'pred_u': load_npy(run_dir, 'dimensional_network_output_u_values.npy'),
        'eta_t': load_npy(run_dir, 'dimensional_zeta_solution_time_mesh_grid.npy'),
        'eta_x': load_npy(run_dir, 'dimensional_zeta_solution_x_mesh_grid.npy'),
        'u_t': load_npy(run_dir, 'dimensional_u_solution_time_mesh_grid.npy'),
        'u_x': load_npy(run_dir, 'dimensional_u_solution_x_mesh_grid.npy'),
    }


def make_metric_record(run_name: str, label: str) -> dict[str, Any]:
    fields = load_run_fields(run_name)
    eta_mse = float(np.mean((fields['pred_eta'] - fields['exact_eta']) ** 2))
    u_mse = float(np.mean((fields['pred_u'] - fields['exact_u']) ** 2))
    return {
        'run_name': run_name,
        'Model': label,
        'MSE_u': u_mse,
        'MSE_eta': eta_mse,
        'L2_u': relative_l2(fields['pred_u'], fields['exact_u']),
        'L2_eta': relative_l2(fields['pred_eta'], fields['exact_eta']),
        'L2_late_u': relative_l2_after_time(
            fields['pred_u'], fields['exact_u'], fields['u_t'], 2.0 * 86400.0
        ),
        'L2_late_eta': relative_l2_after_time(
            fields['pred_eta'], fields['exact_eta'], fields['eta_t'], 2.0 * 86400.0
        ),
        'Combined_L2': combined_relative_l2(
            fields['pred_eta'], fields['exact_eta'], fields['pred_u'], fields['exact_u']
        ),
    }


def metrics_for_runs(run_labels: OrderedDict[str, str]) -> pd.DataFrame:
    missing = [run_name for run_name in run_labels if not (TRAINING_RESULTS_DIR / run_name).is_dir()]
    if missing:
        raise FileNotFoundError(f'Missing run directories: {missing}')
    return pd.DataFrame([make_metric_record(run_name, label) for run_name, label in run_labels.items()])


LATEX_TABLE_COLUMNS = {
    'Model': 'Метод',
    'MSE_u': r'$\mathrm{MSE}_{u}$',
    'MSE_eta': r'$\mathrm{MSE}_{\eta}$',
    'L2_u': r'$E_{2}(u)$',
    'L2_eta': r'$E_{2}(\eta)$',
    'L2_late_u': r'$E_{2}^{t\geq 2d}(u)$',
    'L2_late_eta': r'$E_{2}^{t\geq 2d}(\eta)$',
    'Epochs': 'Число эпох',
    'Training_hours': 'Полное время, ч',
    'Mean_epoch_seconds': 'Среднее время эпохи, с',
}


def scientific_latex(value: float) -> str:
    if value == 0.0:
        return r'$0.00\cdot10^{0}$'
    exponent = int(math.floor(math.log10(abs(value))))
    mantissa = value / (10.0 ** exponent)
    return rf'${mantissa:.2f}\cdot10^{{{exponent}}}$'


def save_latex_table(df: pd.DataFrame, stem: str, scientific: bool = True) -> Path:
    path = TABLE_DIR / f'{stem}.tex'
    latex_df = df.rename(columns=LATEX_TABLE_COLUMNS)
    float_format = scientific_latex if scientific else (lambda value: f'{value:.2f}')
    table = latex_df.to_latex(index=False, float_format=float_format, escape=False)
    path.write_text(table, encoding='utf-8')
    print(f'Saved table: {path.relative_to(PROJECT_ROOT)}')
    return path


def save_pgf(fig: plt.Figure, stem: str) -> Path:
    path = FIGURE_DIR / f'{stem}.pgf'
    fig.savefig(path, bbox_inches='tight')
    print(f'Saved figure: {path.relative_to(PROJECT_ROOT)}')
    return path


def display_and_save_table(
    df: pd.DataFrame,
    stem: str,
    scientific: bool = True,
) -> pd.DataFrame:
    save_latex_table(df, stem, scientific=scientific)
    display(df)
    return df

## Plot helpers

In [ ]:
def plot_training_loss_histories(run_labels: OrderedDict[str, str], stem: str) -> pd.DataFrame:
    rows = []
    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    for run_name, label in run_labels.items():
        run_dir = TRAINING_RESULTS_DIR / run_name
        meta = load_json(run_dir / 'Hyper_Parameter_Dictionary.json')
        loss = load_npy(run_dir, 'total_MSE_over_training.npy', required=False)
        if loss is None:
            continue
        loss = np.asarray(loss, dtype=float).reshape(-1)
        epoch = infer_epoch_axis(loss, meta)
        ax.plot(epoch, loss, linewidth=1.5, label=label)
        rows.extend({
            'run_name': run_name,
            'Model': label,
            'epoch': float(e),
            'LOSS': float(v),
        } for e, v in zip(epoch, loss))
    ax.set_xlabel('Эпоха')
    ax.set_ylabel('Функция потерь')
    ax.set_yscale('log')
    ax.set_ylim(top=10.0)
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(fontsize=8)
    save_pgf(fig, stem)
    plt.show()
    return pd.DataFrame(rows)


def image_extent(t_mesh: np.ndarray, x_mesh: np.ndarray, time_scale: float) -> tuple[float, float, float, float]:
    t_axis = t_mesh[0, :] / time_scale
    x_axis = x_mesh[:, 0] / 1000.0
    return (float(t_axis.min()), float(t_axis.max()), float(x_axis.min()), float(x_axis.max()))


def corrected_triptych(run_name: str, label: str, stem: str) -> None:
    fields = load_run_fields(run_name)
    eta_exact = fields['exact_eta']
    eta_pred = fields['pred_eta']
    u_exact = fields['exact_u']
    u_pred = fields['pred_u']
    eta_err = np.abs(eta_exact - eta_pred)
    u_err = np.abs(u_exact - u_pred)

    eta_value_min = min(0.0, float(np.nanmin(eta_exact)), float(np.nanmin(eta_pred)))
    eta_value_max = max(float(np.nanmax(eta_exact)), float(np.nanmax(eta_pred)))
    eta_err_max = max(float(np.nanmax(eta_err)), np.finfo(float).eps)
    u_abs_max = max(float(np.nanmax(np.abs(u_exact))), float(np.nanmax(np.abs(u_pred))), np.finfo(float).eps)
    u_err_max = max(float(np.nanmax(u_err)), np.finfo(float).eps)

    fig, axes = plt.subplots(2, 3, figsize=(10.6, 6.0), constrained_layout=True)
    fig.suptitle(label, y=1.02)

    columns = ['Численное решение', 'Предсказание сети', 'Абсолютная ошибка']
    for ax, title in zip(axes[0], columns):
        ax.set_title(title)

    eta_panels = [
        (eta_exact, 'viridis', colors.Normalize(vmin=eta_value_min, vmax=eta_value_max), r'$\eta_{\mathrm{числ}}(x,t)$, м'),
        (eta_pred, 'viridis', colors.Normalize(vmin=eta_value_min, vmax=eta_value_max), r'$\eta_{\mathrm{сеть}}(x,t)$, м'),
        (eta_err, 'magma', colors.Normalize(vmin=0.0, vmax=eta_err_max), r'$|\eta_{\mathrm{числ}}-\eta_{\mathrm{сеть}}|$, м'),
    ]
    u_panels = [
        (u_exact, 'coolwarm', colors.TwoSlopeNorm(vcenter=0.0, vmin=-u_abs_max, vmax=u_abs_max), r'$u_{\mathrm{числ}}(x,t)$, м/с'),
        (u_pred, 'coolwarm', colors.TwoSlopeNorm(vcenter=0.0, vmin=-u_abs_max, vmax=u_abs_max), r'$u_{\mathrm{сеть}}(x,t)$, м/с'),
        (u_err, 'magma', colors.Normalize(vmin=0.0, vmax=u_err_max), r'$|u_{\mathrm{числ}}-u_{\mathrm{сеть}}|$, м/с'),
    ]

    eta_extent = image_extent(fields['eta_t'], fields['eta_x'], time_scale=86400.0)
    u_extent = image_extent(fields['u_t'], fields['u_x'], time_scale=86400.0)

    for row, panels, extent in [(0, eta_panels, eta_extent), (1, u_panels, u_extent)]:
        for col, (values, cmap, norm, cbar_label) in enumerate(panels):
            ax = axes[row, col]
            im = ax.imshow(values, origin='lower', aspect='auto', extent=extent, cmap=cmap, norm=norm)
            ax.set_xlabel('Время, дни')
            if col == 0:
                ax.set_ylabel(r'Координата $x$, км')
            fig.colorbar(im, ax=ax, label=cbar_label)

    save_pgf(fig, stem)
    plt.show()


def plot_amplitude_comparison(run_labels: OrderedDict[str, str], stem: str) -> None:
    first_fields = load_run_fields(next(iter(run_labels)))
    t_days = first_fields['eta_t'][0, :] / 86400.0
    eta_ref_amp = np.nanmax(np.abs(first_fields['exact_eta']), axis=0)
    u_ref_amp = np.nanmax(np.abs(first_fields['exact_u']), axis=0)

    fig, axes = plt.subplots(2, 1, figsize=(6.8, 6.4), sharex=True, constrained_layout=True)
    axes[0].plot(t_days, eta_ref_amp, color='black', linewidth=1.8, label=r'Численное решение, $\eta$')
    axes[1].plot(t_days, u_ref_amp, color='black', linewidth=1.8, label=r'Численное решение, $u$')

    for run_name, label in run_labels.items():
        fields = load_run_fields(run_name)
        axes[0].plot(t_days, np.nanmax(np.abs(fields['pred_eta']), axis=0), linewidth=1.4, label=fr'{label} $\eta$')
        axes[1].plot(t_days, np.nanmax(np.abs(fields['pred_u']), axis=0), linewidth=1.4, label=fr'{label} $u$')

    axes[0].set_ylabel(r'$\max |\eta|$, м')
    axes[1].set_ylabel(r'$\max |u|$, м/с')
    axes[1].set_xlabel('Время, дни')
    axes[0].set_title(r'Максимальное абсолютное значение $\eta$')
    axes[1].set_title(r'Максимальное абсолютное значение $u$')
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    save_pgf(fig, stem)
    plt.show()


def plot_spatial_slices(run_name: str, label: str, variable: str, stem: str, time_days=(0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0)) -> None:
    fields = load_run_fields(run_name)
    if variable == 'eta':
        exact = fields['exact_eta']
        pred = fields['pred_eta']
        x = fields['eta_x'][:, 0] / 1000.0
        t = fields['eta_t'][0, :] / 86400.0
        y_label = r'$\eta$, м'
        title_var = r'$\eta$'
    elif variable == 'u':
        exact = fields['exact_u']
        pred = fields['pred_u']
        x = fields['u_x'][:, 0] / 1000.0
        t = fields['u_t'][0, :] / 86400.0
        y_label = r'$u$, м/с'
        title_var = r'$u$'
    else:
        raise ValueError(variable)

    valid_days = [day for day in time_days if float(np.nanmin(t)) <= day <= float(np.nanmax(t))]
    ncols = 2
    nrows = int(math.ceil(len(valid_days) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(8.0, 2.8 * nrows), squeeze=False, constrained_layout=True)
    for ax, day in zip(axes.ravel(), valid_days):
        idx = int(np.nanargmin(np.abs(t - day)))
        ax.plot(x, exact[:, idx], linewidth=1.4, label='Численное решение')
        ax.plot(x, pred[:, idx], linewidth=1.2, label='Предсказание сети')
        ax.set_title(f'{title_var} при t={t[idx]:.3f} дня')
        ax.set_xlabel(r'Координата $x$, км')
        ax.set_ylabel(y_label)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    for ax in axes.ravel()[len(valid_days):]:
        ax.axis('off')
    fig.suptitle(label, y=1.01)
    save_pgf(fig, stem)
    plt.show()

## 1. 1000-epoch experiments

In [ ]:
METRIC_COLUMNS = [
    'Model', 'MSE_u', 'MSE_eta', 'L2_u', 'L2_eta', 'L2_late_u', 'L2_late_eta'
]

all_1k_metrics = metrics_for_runs(RUNS_1K)[METRIC_COLUMNS]
display_and_save_table(all_1k_metrics, 'all_1k_metrics')

In [ ]:
all_1k_loss_history = plot_training_loss_histories(
    RUNS_1K,
    stem='all_1k_training_loss',
)

In [ ]:
for run_name, label in RUNS_1K.items():
    corrected_triptych(
        run_name,
        label,
        stem=f'all_1k_triptych_{run_name}',
    )

## 2. 2000-epoch PINN vs Dynamic PINN

In [ ]:
comparison_2k_metrics = metrics_for_runs(RUNS_2K)[METRIC_COLUMNS]
display_and_save_table(comparison_2k_metrics, 'baseline_dynamic_2k_metrics')

timing_rows = []
for run_name, label in [*RUNS_1K.items(), *RUNS_2K.items()]:
    run_dir = TRAINING_RESULTS_DIR / run_name
    meta = load_json(run_dir / 'Hyper_Parameter_Dictionary.json')
    epoch_times = load_npy(run_dir, 'time_per_epoch.npy')
    timing_rows.append({
        'Model': label,
        'Epochs': int(meta['epochs']),
        'Training_hours': float(meta['computation_time']) / 3600.0,
        'Mean_epoch_seconds': float(np.mean(epoch_times)),
    })
training_times = pd.DataFrame(timing_rows)
display_and_save_table(training_times, 'training_times', scientific=False)

In [ ]:
comparison_2k_loss_history = plot_training_loss_histories(
    RUNS_2K,
    stem='baseline_dynamic_2k_training_loss',
)

In [ ]:
for run_name, label in RUNS_2K.items():
    corrected_triptych(
        run_name,
        label,
        stem=f'baseline_dynamic_2k_triptych_{run_name}',
    )

In [ ]:
plot_amplitude_comparison(
    RUNS_2K,
    stem='baseline_dynamic_2k_amplitude_comparison',
)

In [ ]:
for run_name, label in RUNS_2K.items():
    plot_spatial_slices(
        run_name,
        label,
        variable='eta',
        stem=f'baseline_dynamic_2k_spatial_slices_{run_name}_eta',
    )
    plot_spatial_slices(
        run_name,
        label,
        variable='u',
        stem=f'baseline_dynamic_2k_spatial_slices_{run_name}_u',
    )

## 3. Dynamic PINN vs uploaded checkpoint

In [ ]:
class ArticleMLP(nn.Module):
    def __init__(self, state_dict: OrderedDict[str, torch.Tensor]):
        super().__init__()
        layer_indices = sorted({
            int(key.split('.')[1])
            for key in state_dict
            if key.startswith('layers.') and key.endswith('.weight')
        })
        self.layers = nn.ModuleList([
            nn.Linear(*reversed(state_dict[f'layers.{idx}.weight'].shape))
            for idx in layer_indices
        ])
        self.load_state_dict(state_dict, strict=True)

    def forward(self, t: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        y = torch.cat((t, x), dim=1)
        for layer in self.layers[:-1]:
            y = torch.tanh(layer(y))
        return self.layers[-1](y)


def find_article_best_state_path(article_dir: Path) -> Path:
    paths = sorted(p for p in article_dir.glob('Best_State_Dict*') if not p.name.endswith(':Zone.Identifier'))
    if not paths:
        raise FileNotFoundError(f'No Best_State_Dict checkpoint found in {article_dir}')
    return paths[0]


def evaluate_article_checkpoint(state_path: Path, template_run: str) -> dict[str, Any]:
    template_dir = TRAINING_RESULTS_DIR / template_run
    meta_path = ARTICLE_RUN_DIR / 'Hyper_Parameter_Dictionary.json'
    meta = load_json(meta_path) if meta_path.exists() else {}
    vertical_length_scale = float(meta.get('vertical_length_scale', 13.655371118466222))
    horizontal_length_scale = float(meta.get('horizontal_length_scale', 1_000_000.0))
    time_scale = float(meta.get('time_scale', 86400.0))

    state_dict = safe_torch_load(state_path, map_location='cpu')
    if not isinstance(state_dict, OrderedDict):
        state_dict = OrderedDict(state_dict)
    state_dict = OrderedDict((key, value.detach().float().cpu()) for key, value in state_dict.items())
    model = ArticleMLP(state_dict).to(DEVICE).eval()

    exact_eta = load_npy(template_dir, 'exact_solution_h_values.npy')
    exact_u = load_npy(template_dir, 'exact_solution_u_values.npy')
    zeta_t = load_npy(template_dir, 'zeta_solution_time_input_grid.npy').astype('float32')
    zeta_x = load_npy(template_dir, 'zeta_solution_x_input_grid.npy').astype('float32')
    u_t = load_npy(template_dir, 'u_solution_time_input_grid.npy').astype('float32')
    u_x = load_npy(template_dir, 'u_solution_x_input_grid.npy').astype('float32')

    def predict(t_np: np.ndarray, x_np: np.ndarray, output_col: int, scale: float, shape: tuple[int, int]) -> np.ndarray:
        chunks = []
        with torch.no_grad():
            for start in range(0, len(t_np), ARTICLE_EVAL_BATCH_SIZE):
                end = min(start + ARTICLE_EVAL_BATCH_SIZE, len(t_np))
                t = torch.from_numpy(t_np[start:end]).to(DEVICE)
                x = torch.from_numpy(x_np[start:end]).to(DEVICE)
                y = model(t, x)[:, output_col:output_col + 1].detach().cpu().numpy()
                chunks.append(y)
        return (np.vstack(chunks).reshape(shape) * scale).astype('float32')

    pred_eta = predict(zeta_t, zeta_x, output_col=1, scale=vertical_length_scale, shape=exact_eta.shape)
    pred_u = predict(u_t, u_x, output_col=0, scale=horizontal_length_scale / time_scale, shape=exact_u.shape)

    return {
        'Model': 'PINN Demir, Logemann и Greenberg',
        'L2_u': relative_l2(pred_u, exact_u),
        'L2_eta': relative_l2(pred_eta, exact_eta),
        'Combined_L2': combined_relative_l2(pred_eta, exact_eta, pred_u, exact_u),
    }


article_checkpoint_path = find_article_best_state_path(ARTICLE_RUN_DIR)
dynamic_fields = load_run_fields(DYNAMIC_2K_RUN)
dynamic_l2_record = {
    'Model': 'PINN с модальной динамической компонентой',
    'L2_u': relative_l2(dynamic_fields['pred_u'], dynamic_fields['exact_u']),
    'L2_eta': relative_l2(dynamic_fields['pred_eta'], dynamic_fields['exact_eta']),
    'Combined_L2': combined_relative_l2(
        dynamic_fields['pred_eta'], dynamic_fields['exact_eta'], dynamic_fields['pred_u'], dynamic_fields['exact_u']
    ),
}
checkpoint_l2_record = evaluate_article_checkpoint(article_checkpoint_path, DYNAMIC_2K_RUN)
dynamic_vs_checkpoint_l2 = pd.DataFrame([dynamic_l2_record, checkpoint_l2_record])[
    ['Model', 'L2_u', 'L2_eta']
]

display_and_save_table(dynamic_vs_checkpoint_l2, 'dynamic_2k_vs_checkpoint_l2')